# Kaggle submission driver

The final submission (four members, 0.25 each) run from this repo's code.
Attach: the competition data; `tuckerarrants/smp-offline` (wheels); `tuckerarrants/rogii-v2s`
and the `tuckerarrants/rogii-l2` model (checkpoints). For the code, either attach this repo as a dataset
(`python scripts/make_kaggle_src.py` builds it) or turn internet on and the next cell clones it from GitHub.

**As a check.** Run it interactively (not as a submission): Kaggle then shows the 3 test wells outside
scoring, and the last cell compares the result with the numbers the notebook of record printed for those
wells. Expect every value to match to about 0.001 ft. Roughly 15–30 minutes on 2× T4, most of it loading checkpoints.

In [ ]:
!pip install -q /kaggle/input/datasets/tuckerarrants/smp-offline/smp_offline_wheels/smp_offline_wheels/segmentation_models_pytorch-0.5.0-py3-none-any.whl
!pip install -q /kaggle/input/datasets/tuckerarrants/smp-offline/smp_offline_wheels/smp_offline_wheels/timm-1.0.27-py3-none-any.whl

In [ ]:
import glob, os, subprocess, sys
hits = sorted(glob.glob('/kaggle/input/**/rogii_wellbore/__init__.py', recursive=True))
if hits:                                   # the code dataset is attached
    REPO = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    sys.path.insert(0, os.path.join(REPO, 'src'))
else:                                      # internet on: clone the repo
    REPO = '/kaggle/working/rogii-wellbore-2026'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/TuckerArrants/rogii-wellbore-2026.git', REPO], check=True)
    sys.path.insert(0, os.path.join(REPO, 'src'))
CONFIG = os.path.join(REPO, 'configs/submit/final_equal_weight.yaml')
COMP = next(p for p in ['/kaggle/input/competitions/rogii-wellbore-geology-prediction',
                        '/kaggle/input/rogii-wellbore-geology-prediction'] if os.path.exists(p))

from rogii_wellbore.infer import run
sub = run(CONFIG, COMP, 'submission.csv')
sub.head()

In [ ]:
# Check against the notebook of record (only meaningful on the 3 visible test wells)
import json
from rogii_wellbore.infer import check_reference
ref = json.load(open(os.path.join(REPO, 'pipeline/03_submit/reference_rows.json')))
print('PASS' if check_reference(sub, ref) else 'FAIL')